# Analyse de sentiments sur des tweets

Ce TP suit celui des auteurs (« Classification de texte : retrouver l'auteur d'une phrase »). Nous y classons des tweets en trois catégories, **négatif**, **neutre** ou **positif**, sur le jeu de données [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) (tâche *sentiment*) : environ 60 000 tweets en anglais, annotés à la main.

La préparation du texte, le classifieur récurrent et son entraînement avec Lightning sont ceux du TP des auteurs, sous les mêmes noms (`Tokenizer`, `vectorise`, `GRUTexte`, `ClassifieurTexte`) : ils sont donnés. Ce TP s'attache à ce qui change :

1. Des classes déséquilibrées, et le rappel moyen pour les mesurer.
2. Des batchs sans aucun *padding* : `pack_sequence`.
3. Un GRU bidirectionnel qui lit des séquences empaquetées.
4. L'entraînement et l'évaluation du GRU.
5. Un transformeur déjà affiné sur la tâche.
6. Affiner soi-même un encodeur pré-entraîné, ou n'entraîner que sa tête.

Les parties 5 et 6 utilisent des transformeurs, le sujet du chapitre suivant.

Pensez à activer le GPU : Exécution > Modifier le type d'exécution > GPU. Sans lui, l'entraînement de la partie 4 est plusieurs fois plus lent, et l'affinage de la partie 6 prend des heures.

## Installation et import des bibliothèques

In [ ]:
!pip install -q datasets lightning torchmetrics

In [ ]:
import collections
from collections.abc import Iterable

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import PackedSequence, pack_sequence
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassRecall

lightning.seed_everything(0)

## Les données

La bibliothèque `datasets` de Hugging Face télécharge le jeu de données déjà découpé en entraînement, validation et test. La cellule d'après affiche dix tweets d'entraînement avec leur label, la répartition des classes et la longueur des tweets (en mots).

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
noms_etiquettes = ["négatif", "neutre", "positif"]
print(tweets)

In [ ]:
df_entrainement = tweets["train"].to_pandas()
df_entrainement["sentiment"] = df_entrainement.label.map(
  dict(enumerate(noms_etiquettes))
)
print(df_entrainement.sample(10, random_state=0)[["sentiment", "text"]].to_string())

df_entrainement.sentiment.value_counts().plot.bar(title="Répartition des classes")
plt.show()

df_entrainement.text.str.split().str.len().plot.hist(
  bins=30, title="Longueur des tweets (mots)"
)
plt.show()

## 1. Des classes déséquilibrées

Comment ont été traités les noms d'utilisateurs ? Les classes sont-elles équilibrées, et quelle conséquence pour le choix de la métrique ? Quelle est la longueur typique d'un tweet ?

Votre réponse ici

### Solution

- Les noms d'utilisateurs sont remplacés par `@user` : les tweets sont anonymisés.
- La classe *neutre* est la plus fréquente, la classe *négatif* la plus rare : la justesse (*accuracy*) favoriserait un modèle qui prédit surtout *neutre*. La métrique officielle de TweetEval est le **rappel moyen par classe** (*macro recall*), qui donne le même poids à chaque classe.
- Un tweet fait une vingtaine de mots, rarement plus de 40.

### Une référence : TF-IDF et régression logistique

Avant d'entraîner un réseau de neurones, mesurons ce qu'obtient un modèle simple : une régression logistique sur une représentation TF-IDF des tweets. Si le réseau ne fait pas mieux, il est inutile !

In [ ]:
vectoriseur = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectoriseur.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectoriseur.transform(tweets["validation"]["text"])

reference = LogisticRegression(max_iter=1000)
reference.fit(X_train_tfidf, tweets["train"]["label"])
rappel_reference = recall_score(
  tweets["validation"]["label"], reference.predict(X_val_tfidf), average="macro"
)
print(f"Rappel moyen TF-IDF en validation : {rappel_reference:.3f}")

## La préparation du TP des auteurs

La classe `Tokenizer` est celle du TP des auteurs, à une différence près : elle découpe le texte en mots avec spaCy (`spacy.blank("en")`, un pipeline anglais qui ne contient que le tokeniseur, très rapide), qui garde la ponctuation et les émoticônes, porteuses de sentiment dans un tweet, et sépare les contractions (« don't » → « do », « n't »). Comme dans ce TP, `tokenizer` apprend les 20 000 mots de son vocabulaire sur les seuls tweets d'entraînement : `<pad>` (indice 0, inutilisé ici), `<unk>` (indice 1), puis les mots du plus au moins fréquent.

`vectorise(textes)` donne la séquence d'indices de chaque texte, comme dans le TP des auteurs, mais sans troncature ni *padding* : une liste de tenseurs de longueurs différentes, que la partie 2 empaquette. `donnees_ent`, `donnees_val` et `donnees_test` sont les listes des couples `(séquence d'indices, label)`.

In [ ]:
nlp = spacy.blank("en")


class Tokenizer:
  """Numérote les mots les plus fréquents d'un corpus, à partir de 2."""

  def __init__(self, max_jetons: int) -> None:
    self.max_jetons = max_jetons
    self.vocabulaire: list[str] = []
    self.indice_mots: dict[str, int] = {}

  def decoupe(self, texte: str) -> list[str]:
    # Découpage de spaCy : ponctuation et émoticônes gardées
    return [jeton.lower_ for jeton in nlp.tokenizer(texte) if not jeton.is_space]

  def apprend_vocabulaire(self, textes: Iterable[str]) -> None:
    comptes = collections.Counter(
      mot for texte in textes for mot in self.decoupe(texte)
    )
    # Indices 0 et 1 réservés au padding et aux mots inconnus
    plus_frequents = [mot for mot, _ in comptes.most_common(self.max_jetons - 2)]
    self.vocabulaire = ["<pad>", "<unk>", *plus_frequents]
    self.indice_mots = {mot: index for index, mot in enumerate(self.vocabulaire)}

  def convertit_en_sequences(self, textes: Iterable[str]) -> list[list[int]]:
    # Un mot hors du vocabulaire devient 1, <unk>
    return [
      [self.indice_mots.get(mot, 1) for mot in self.decoupe(texte)] for texte in textes
    ]


tokenizer = Tokenizer(max_jetons=20_000)
# Vocabulaire de l'entraînement seul
tokenizer.apprend_vocabulaire(tweets["train"]["text"])
vocabulaire = tokenizer.vocabulaire


def vectorise(textes: list[str]) -> list[torch.Tensor]:
  # Ni troncature ni padding. Un tweet sans aucun token devient un
  # unique <unk> : une séquence vide ne peut pas être empaquetée
  return [
    torch.tensor(sequence or [1])
    for sequence in tokenizer.convertit_en_sequences(textes)
  ]


# Couples (séquence d'indices, label) : une liste suffit comme Dataset
donnees_ent = list(zip(vectorise(tweets["train"]["text"]), tweets["train"]["label"]))
donnees_val = list(
  zip(vectorise(tweets["validation"]["text"]), tweets["validation"]["label"])
)
donnees_test = list(zip(vectorise(tweets["test"]["text"]), tweets["test"]["label"]))
print(" | ".join(vocabulaire[:12]))
print(tweets["train"][0]["text"])
print(donnees_ent[0])

## 2. Des batchs sans *padding* : `pack_sequence`

Les tweets n'ont pas tous la même longueur. Plutôt que de compléter les séquences d'un batch avec du *padding*, comme `vectorise` le fait dans le TP des auteurs, PyTorch peut les **empaqueter** dans une `PackedSequence` : tous les tokens du batch sont rangés bout à bout, pas de temps par pas de temps, sans aucun *padding*.

- `data` : d'abord le premier token de chaque séquence, puis le deuxième token de chaque séquence qui en a au moins deux, etc. ;
- `batch_sizes` : le nombre de séquences encore en cours à chaque pas de temps ;
- `sorted_indices` / `unsorted_indices` : l'ordre (par longueur décroissante) dans lequel les séquences sont rangées, et de quoi revenir à l'ordre d'origine.

Les couches récurrentes de PyTorch (`nn.RNN`, `nn.GRU`, `nn.LSTM`) acceptent directement une `PackedSequence` : à chaque pas de temps, elles ne calculent que sur les séquences encore en cours, et leur dernier état caché `h_n` est celui du dernier *vrai* token de chaque séquence, remis dans l'ordre d'origine. Dans le TP des auteurs, `pack_padded_sequence` construisait cette `PackedSequence` à partir d'un tenseur avec *padding* ; `pack_sequence` la construit directement à partir des séquences.

Exécutez la cellule suivante, puis retrouvez comment `data`, `batch_sizes` et `sorted_indices` sont construits à partir des trois séquences.

In [ ]:
exemple = pack_sequence(
  [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6, 7, 8, 9])],
  enforce_sorted=False,
)
print(exemple)

Votre réponse ici

### Solution

- `sorted_indices` vaut `[2, 0, 1]` : les séquences sont rangées de la plus longue à la plus courte, `[6, 7, 8, 9]` (la n° 2), `[1, 2, 3]` (la n° 0), puis `[4, 5]` (la n° 1).
- `data` vaut `[6, 1, 4, 7, 2, 5, 8, 3, 9]` : les premiers tokens des trois séquences, dans cet ordre, puis leurs deuxièmes tokens, puis les troisièmes des deux séquences qui en ont, puis le quatrième de la plus longue.
- `batch_sizes` vaut `[3, 3, 2, 1]` : le nombre de séquences encore en cours à chacun des quatre pas de temps. Sa longueur est celle de la plus longue séquence, et sa somme le nombre de tokens.

### La fonction `assemble_lot`

Écrivez la fonction `assemble_lot(batch)` que les `DataLoader` utiliseront pour former leurs batchs : elle reçoit une liste de couples `(tenseur d'indices, label)`, ceux de `donnees_ent`, et renvoie le couple `(séquences empaquetées, tenseur des labels)`. Les séquences d'un batch ne sont pas triées par longueur : `pack_sequence` doit s'en charger.

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_sequence.html

In [ ]:
def assemble_lot(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  # Votre code ici
  ...

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sequences, etiquettes = zip(*batch)` : un tuple des séquences et un tuple des labels
- `pack_sequence(sequences, enforce_sorted=False)` : `enforce_sorted=False` laisse `pack_sequence` trier les séquences par longueur
- `torch.tensor(etiquettes)` : le tenseur des labels, que l'entropie croisée attend
- `assemble_lot([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)])` renvoie une `PackedSequence` de `data` `[1, 4, 2, 5, 3]` et de `batch_sizes` `[2, 2, 1]`, et les labels `tensor([0, 2])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def assemble_lot(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  sequences, etiquettes = zip(*...)
  return pack_sequence(..., enforce_sorted=...), torch.tensor(...)


print(assemble_lot([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)]))
```

#### Solution

In [ ]:
def assemble_lot(
  batch: list[tuple[torch.Tensor, int]],
) -> tuple[PackedSequence, torch.Tensor]:
  # Un tuple des séquences, un tuple des labels
  sequences, etiquettes = zip(*batch)
  # pack_sequence trie lui-même les séquences par longueur
  return pack_sequence(sequences, enforce_sorted=False), torch.tensor(etiquettes)


print(assemble_lot([(torch.tensor([1, 2, 3]), 0), (torch.tensor([4, 5]), 2)]))

### Des batchs empaquetés

Les `DataLoader` forment leurs batchs avec `assemble_lot`. Comparons, sur un batch d'entraînement, le nombre de tokens empaquetés à la taille qu'aurait le même batch complété avec du *padding* :

In [ ]:
chargeur_ent = DataLoader(
  donnees_ent, batch_size=64, shuffle=True, collate_fn=assemble_lot
)
chargeur_val = DataLoader(donnees_val, batch_size=64, collate_fn=assemble_lot)
chargeur_test = DataLoader(donnees_test, batch_size=64, collate_fn=assemble_lot)

empaquetees, etiquettes = next(iter(chargeur_ent))
# Nombre de tweets × longueur du plus long
taille_avec_padding = len(etiquettes) * len(empaquetees.batch_sizes)
print(
  f"Tokens du batch : {empaquetees.data.numel()} empaquetés contre {taille_avec_padding} "
  f"avec padding, soit {1 - empaquetees.data.numel() / taille_avec_padding:.0%} de padding évité"
)

Les longueurs des tweets d'un batch varient beaucoup : complété avec du *padding*, un batch contiendrait environ deux fois plus de positions que de vrais tokens, et le GRU ferait autant de calculs inutiles. La `PackedSequence` les évite.

## 3. Un GRU bidirectionnel sur des séquences empaquetées

`GRUTexte` a les couches de celui du TP des auteurs : `embedding`, un vecteur de 64 nombres par mot du vocabulaire, un *dropout* de taux `dropout`, `gru`, un GRU bidirectionnel de `nb_unites` unités par sens, et `tete`, qui donne un score (un logit) à chacune des 3 classes à partir des derniers états cachés des deux sens. Mais il reçoit une `PackedSequence` d'indices, et non plus un tenseur avec *padding* : il n'a plus besoin de `pack_padded_sequence`.

Écrivez sa méthode `forward(empaquetees)`, qui renvoie les logits des tweets du batch, de forme `(batch, 3)`, avec le *dropout* sur les embeddings et avant `tete`. `nn.Embedding` et `nn.Dropout` n'acceptent pas de `PackedSequence` : appliquez-les aux tokens `empaquetees.data`, puis reconstruisez une `PackedSequence` de même structure avec `empaquetees._replace(data=...)`, qui garde `batch_sizes` et les indices de tri.

Pourquoi `h_n[-2]` et `h_n[-1]` : ce que contient `h_n` à la fin d’un GRU ou d’un LSTM, à plusieurs couches et bidirectionnel, en vidéo : [`h_n` couche par couche](https://shuuchuu.github.io/videos/nn/gru-final-hidden-state.mp4).

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html
- https://docs.python.org/3/library/collections.html#collections.somenamedtuple._replace

In [ ]:
class GRUTexte(nn.Module):
  def __init__(self, nb_unites: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulaire), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, nb_unites, bidirectional=True)
    self.tete = nn.Linear(2 * nb_unites, len(noms_etiquettes))

  def forward(self, empaquetees: PackedSequence) -> torch.Tensor:
    # Votre code ici
    ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `empaquetees.data` : les indices de tous les tokens du batch, de forme `(tokens,)` ; `self.embedding(empaquetees.data)` est de forme `(tokens, 64)`
- `empaquetees._replace(data=...)` : une nouvelle `PackedSequence`, de mêmes `batch_sizes` et indices de tri, avec d'autres `data`
- `_, h_n = self.gru(vecteurs)` : `h_n` a la forme `(2, batch, nb_unites)`, ses lignes dans l'ordre d'origine des tweets ; `h_n[-2]` et `h_n[-1]` sont les derniers états des sens avant et arrière
- `torch.cat([h_n[-2], h_n[-1]], dim=1)` les met bout à bout, en un tenseur de forme `(batch, 2 * nb_unites)`
- `GRUTexte()(assemble_lot(donnees_ent[:4])[0]).shape` vaut `torch.Size([4, 3])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class GRUTexte(nn.Module):
  def __init__(self, nb_unites: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulaire), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, nb_unites, bidirectional=True)
    self.tete = nn.Linear(2 * nb_unites, len(noms_etiquettes))

  def forward(self, empaquetees: PackedSequence) -> torch.Tensor:
    vecteurs = empaquetees._replace(data=self.dropout(self.embedding(...)))
    _, h_n = self.gru(...)
    resume = torch.cat([..., ...], dim=1)
    return self.tete(self.dropout(...))


print(GRUTexte()(assemble_lot(donnees_ent[:4])[0]).shape)
```

### Solution

In [ ]:
class GRUTexte(nn.Module):
  """Embeddings, GRU bidirectionnel sur des tweets empaquetés, un score par classe."""

  def __init__(self, nb_unites: int = 64, dropout: float = 0.3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(vocabulaire), 64, padding_idx=0)
    self.dropout = nn.Dropout(dropout)
    self.gru = nn.GRU(64, nb_unites, bidirectional=True)
    self.tete = nn.Linear(2 * nb_unites, len(noms_etiquettes))

  def forward(self, empaquetees: PackedSequence) -> torch.Tensor:
    # Embeddings et dropout de tous les tokens du batch à la fois, remis dans
    # une PackedSequence de même structure
    vecteurs = empaquetees._replace(data=self.dropout(self.embedding(empaquetees.data)))
    _, h_n = self.gru(vecteurs)
    # Derniers états cachés des sens avant et arrière, dans l'ordre
    # d'origine des tweets
    resume = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.tete(self.dropout(resume))


print(GRUTexte()(assemble_lot(donnees_ent[:4])[0]).shape)
print(GRUTexte())
print(sum(p.numel() for p in GRUTexte().parameters()), "paramètres")

## 4. Entraînement et évaluation

`ClassifieurTexte` est l'enveloppe Lightning du TP des auteurs, qui mesure le rappel moyen par classe (`MulticlassRecall(average="macro")`) en validation et en test, au lieu de la justesse. Ses batchs sont des couples `(PackedSequence, etiquettes)` : le `Trainer` les déplace lui-même sur le GPU, `PackedSequence` comprise.

L'entraînement est celui du TP des auteurs : 10 epochs au plus, la norme du gradient écrêtée à 1 (`gradient_clip_val`), contre l'explosion du gradient, `EarlyStopping` après 2 epochs sans progrès et `ModelCheckpoint` pour garder les poids de la meilleure epoch. Une différence : la meilleure epoch est celle du meilleur rappel moyen de validation (`monitor="val_recall", mode="max"`), la métrique de la tâche, et non plus celle de la plus faible perte de validation. `trace_metriques` trace ensuite les courbes d'apprentissage.

In [ ]:
class ClassifieurTexte(lightning.LightningModule):
  """Enveloppe Lightning : classe le sentiment de tweets empaquetés."""

  def __init__(self, modele: nn.Module, taux_apprentissage: float = 1e-3) -> None:
    super().__init__()
    self.modele = modele
    self.taux_apprentissage = taux_apprentissage
    # torchmetrics demande une instance de métrique par étape
    self.rappels = nn.ModuleDict(
      {
        f"{stage}_recall": MulticlassRecall(
          num_classes=len(noms_etiquettes), average="macro"
        )
        for stage in ("val", "test")
      }
    )

  def forward(self, empaquetees: PackedSequence) -> torch.Tensor:
    return self.modele(empaquetees)

  def etape(
    self, batch: tuple[PackedSequence, torch.Tensor], stage: str
  ) -> torch.Tensor:
    empaquetees, etiquettes = batch
    logits = self(empaquetees)
    perte = nn.functional.cross_entropy(logits, etiquettes)
    # Sans batch_size, Lightning le déduirait de empaquetees.data : le nombre de tokens
    self.log(
      f"{stage}_loss",
      perte,
      batch_size=len(etiquettes),
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    if stage != "train":
      rappel = self.rappels[f"{stage}_recall"]
      rappel(logits, etiquettes)
      self.log(
        f"{stage}_recall",
        rappel,
        batch_size=len(etiquettes),
        on_step=False,
        on_epoch=True,
        prog_bar=True,
      )
    return perte

  def training_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "train")

  def validation_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "val")

  def test_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "test")

  def predict_step(
    self, batch: tuple[PackedSequence, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    empaquetees, _ = batch
    return self(empaquetees).argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.taux_apprentissage)


def trace_metriques(trainer: lightning.Trainer) -> None:
  """Trace les pertes et le rappel moyen de validation au fil des epochs."""
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par epoch
  metriques = (
    pandas.read_csv(f"{trainer.logger.log_dir}/metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_loss", "val_loss"])
  )
  metriques[["train_loss", "val_loss"]].plot(title="Perte", marker="o")
  metriques[["val_recall"]].plot(
    title="Rappel moyen en validation", marker="o", legend=False
  )
  plt.show()

In [ ]:
modele = ClassifieurTexte(GRUTexte())
# Meilleure epoch : celle du meilleur rappel moyen de validation
point_controle = ModelCheckpoint(monitor="val_recall", mode="max")
trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  gradient_clip_val=1.0,  # écrêtage de la norme du gradient
  logger=CSVLogger("logs", name="sentiment_gru"),
  callbacks=[
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    TQDMProgressBar(),
    EarlyStopping(monitor="val_recall", mode="max", patience=2),
    point_controle,
  ],
)
trainer.fit(modele, chargeur_ent, chargeur_val)
print(f"Meilleur rappel moyen en validation : {point_controle.best_model_score:.3f}")
trace_metriques(trainer)

### Évaluation sur l'ensemble de test

La cellule suivante évalue le meilleur modèle sur l'ensemble de test, puis affiche, avec `affiche_evaluation`, le rapport de classification et la matrice de confusion de ses prédictions et de celles de la référence TF-IDF.

In [ ]:
def affiche_evaluation(
  verites: numpy.ndarray, predictions: numpy.ndarray, titre: str
) -> None:
  """Affiche le rapport de classification et la matrice de confusion."""
  print(titre)
  print(
    classification_report(verites, predictions, target_names=noms_etiquettes, digits=3)
  )
  ConfusionMatrixDisplay.from_predictions(
    verites, predictions, display_labels=noms_etiquettes, normalize="true"
  )
  plt.title(titre)
  plt.show()


trainer.test(modele, chargeur_test, ckpt_path="best")

# trainer.predict renvoie la liste des sorties de predict_step, une par batch
predictions = torch.cat(
  trainer.predict(modele, chargeur_test, ckpt_path="best")
).numpy()
verites = numpy.array(tweets["test"]["label"])
affiche_evaluation(verites, predictions, "GRU bidirectionnel")

predictions_reference = reference.predict(vectoriseur.transform(tweets["test"]["text"]))
affiche_evaluation(verites, predictions_reference, "Référence TF-IDF")

### Les classes confondues

Quelles classes sont le plus souvent confondues ? Le GRU fait-il mieux que la référence TF-IDF ?

Votre réponse ici

#### Solution

Dans notre exécution :

- Les confusions se font surtout avec la classe *neutre* : le GRU prend pour neutres 46 % des tweets négatifs et 32 % des positifs ; *négatif* et *positif* sont rarement confondus entre eux (moins de 10 %).
- Le GRU appris depuis zéro ne fait qu'un peu mieux que la référence TF-IDF : un rappel moyen de 0,59 contre 0,56 en test. Il gagne surtout sur la classe *négatif*, la plus rare à l'entraînement, que la référence classe neutre dans 60 % des cas. Avec quelques dizaines de milliers de tweets seulement, il manque de données pour apprendre le sens des mots : c'est la motivation des modèles pré-entraînés.

## 5. Un transformeur déjà affiné sur la tâche

Le modèle [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) est un RoBERTa pré-entraîné sur des millions de tweets, puis affiné sur cette même tâche. La fonction `pipeline` de `transformers`, que la cellule suivante appelle, permet de l'utiliser en une ligne.

Évaluez-le sur les 1 000 premiers tweets de test, et comparez son rappel moyen à celui du GRU sur ces mêmes tweets. Qu'en concluez-vous ?

**Documentation utile** :

- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline

In [ ]:
from transformers import pipeline

classifieur = pipeline(
  "sentiment-analysis",
  model="cardiffnlp/twitter-roberta-base-sentiment-latest",
  device=0 if torch.cuda.is_available() else -1,
)
print(classifieur("I love this course on PyTorch!"))

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tweets["test"].select(range(1000))` : les 1 000 premiers tweets de test
- `classifieur(list(sous_ensemble["text"]), batch_size=64)` : une liste d'un dictionnaire par tweet, comme `{'label': 'positive', 'score': 0.98}` ; les labels sont `negative`, `neutral` et `positive`, dans l'ordre de `noms_etiquettes`
- `recall_score(verites, predictions, average="macro")` : le rappel moyen ; `predictions` contient les prédictions du GRU sur tout l'ensemble de test, dans l'ordre

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
sous_ensemble = tweets["test"].select(range(...))
nom_vers_etiquette = {"negative": 0, "neutral": 1, "positive": 2}
predictions_roberta = [
  nom_vers_etiquette[sortie[...]]
  for sortie in classifieur(list(sous_ensemble["text"]), batch_size=64)
]
rappel_roberta = recall_score(list(sous_ensemble["label"]), ..., average="macro")
rappel_gru = recall_score(verites[:1000], ...[:1000], average="macro")
print(f"RoBERTa : {rappel_roberta:.3f}, GRU : {rappel_gru:.3f}")
```

### Solution

In [ ]:
sous_ensemble = tweets["test"].select(range(1000))
nom_vers_etiquette = {"negative": 0, "neutral": 1, "positive": 2}
# Le pipeline renvoie un dictionnaire par tweet : {'label': ..., 'score': ...}
predictions_roberta = [
  nom_vers_etiquette[sortie["label"]]
  for sortie in classifieur(list(sous_ensemble["text"]), batch_size=64)
]
rappel_roberta = recall_score(
  list(sous_ensemble["label"]), predictions_roberta, average="macro"
)
rappel_gru = recall_score(verites[:1000], predictions[:1000], average="macro")
print(f"RoBERTa : {rappel_roberta:.3f}, GRU : {rappel_gru:.3f}")

Le transformeur pré-entraîné fait nettement mieux (environ 0,71 contre 0,59 pour le GRU) : il a appris le langage des tweets sur des millions d'exemples avant d'être affiné. Pour une tâche de classification de texte courante, partir d'un modèle pré-entraîné est aujourd'hui le premier réflexe ; entraîner un RNN depuis zéro reste utile quand les contraintes de calcul sont fortes, ou quand aucun modèle pré-entraîné n'existe pour la langue ou le domaine.

## 6. Affiner soi-même un encodeur pré-entraîné

Le RoBERTa de cardiffnlp a été affiné par d'autres sur cette tâche. Faisons-le nous-mêmes, à partir d'un encodeur généraliste qui n'a jamais vu ces labels : [`distilroberta-base`](https://huggingface.co/distilbert/distilroberta-base), une version réduite de RoBERTa (82 M de paramètres). `AutoModelForSequenceClassification` lui ajoute une tête de classification à 3 sorties, aux poids aléatoires (slide « Affiner un encodeur pré-entraîné », au chapitre suivant).

La cellule suivante tokenise les tweets avec le tokeniseur de l'encodeur, et définit :

- `ClassifieurTransformeur(nom, taux_apprentissage)`, l'enveloppe Lightning de l'encodeur et de sa tête ;
- `affine(classifieur, nom)`, qui entraîne un `ClassifieurTransformeur` pendant une epoch, puis renvoie ses prédictions sur les 1 000 premiers tweets de test, ceux de `verites[:1000]`.

In [ ]:
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
)

ENCODEUR = "distilroberta-base"
tokenizer_encodeur = AutoTokenizer.from_pretrained(ENCODEUR)
# Tokenisation de chaque tweet : identifiants des sous-mots et masque d'attention
tweets_encodes = (
  tweets.map(
    lambda batch: tokenizer_encodeur(batch["text"], truncation=True, max_length=128),
    batched=True,
  )
  .remove_columns(["text"])
  .rename_column("label", "labels")
)
# Le padding est fait batch par batch, à la longueur du plus long tweet
assembleur_encodeur = DataCollatorWithPadding(tokenizer_encodeur)
chargeurs_enc = {
  partie: DataLoader(
    tweets_encodes[partie],
    batch_size=32,
    shuffle=partie == "train",
    collate_fn=assembleur_encodeur,
  )
  for partie in ("train", "validation")
}
# Les 1 000 premiers tweets de test, ceux des comparaisons
chargeurs_enc["test"] = DataLoader(
  tweets_encodes["test"].select(range(1000)),
  batch_size=32,
  collate_fn=assembleur_encodeur,
)


class ClassifieurTransformeur(lightning.LightningModule):
  """Encodeur pré-entraîné surmonté d'une nouvelle tête de classification."""

  def __init__(self, nom: str, taux_apprentissage: float = 2e-5) -> None:
    super().__init__()
    self.taux_apprentissage = taux_apprentissage
    # La tête de classification est créée avec des poids aléatoires
    self.modele = AutoModelForSequenceClassification.from_pretrained(
      nom, num_labels=len(noms_etiquettes)
    )
    # from_pretrained rend le modèle en mode évaluation, que Lightning 2.6 ne
    # quitte plus au début de fit : sans train(), il serait affiné sans dropout
    self.modele.train()
    self.rappel_val = MulticlassRecall(
      num_classes=len(noms_etiquettes), average="macro"
    )

  def etape(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
    # Avec batch["labels"], le modèle calcule lui-même l'entropie croisée
    sorties = self.modele(**batch)
    taille_lot = len(batch["labels"])
    self.log(
      f"{stage}_loss",
      sorties.loss,
      batch_size=taille_lot,
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    if stage == "val":
      self.rappel_val(sorties.logits, batch["labels"])
      self.log(
        "val_recall",
        self.rappel_val,
        batch_size=taille_lot,
        on_step=False,
        on_epoch=True,
        prog_bar=True,
      )
    return sorties.loss

  def training_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "train")

  def validation_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "val")

  def predict_step(
    self, batch: dict[str, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.modele(**batch).logits.argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # Les paramètres figés (requires_grad_(False)) n'ont pas de gradient :
    # AdamW les laisse tels quels
    return torch.optim.AdamW(self.parameters(), lr=self.taux_apprentissage)


def affine(classifieur: ClassifieurTransformeur, nom: str) -> numpy.ndarray:
  """Entraîne classifieur une epoch, puis prédit les 1 000 premiers tweets de test."""
  trainer = lightning.Trainer(
    max_epochs=1,
    accelerator="auto",
    devices=1,
    # Calculs en demi-précision sur GPU : deux fois plus rapides
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=CSVLogger("logs", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,  # une seule epoch, gardée telle quelle
  )
  trainer.fit(classifieur, chargeurs_enc["train"], chargeurs_enc["validation"])
  return torch.cat(trainer.predict(classifieur, chargeurs_enc["test"])).cpu().numpy()

### Affiner tout le modèle, ou n'entraîner que sa tête ?

Un encodeur pré-entraîné se réutilise de deux façons :

- l'**affinage** (*fine-tuning*) : tout le modèle apprend, avec un petit learning rate (2e-5) qui préserve ce que l'encodeur a appris pendant son pré-entraînement ;
- l'**extraction de caractéristiques** (*feature extraction*) : l'encodeur est figé, et seule la tête apprend, à partir de ses représentations ; elle part de zéro, avec le learning rate d'un modèle neuf (1e-3).

Comparez-les. Créez deux `ClassifieurTransformeur`, figez l'encodeur du second (`modele.roberta`) avec `requires_grad_`, et comptez les paramètres entraînables de chacun. Entraînez-les avec `affine`, puis comparez leurs rappels moyens sur les 1 000 premiers tweets de test à ceux du GRU et du RoBERTa de cardiffnlp. Quelle démarche l'emporte, et pourquoi ? Laquelle a l'epoch la plus courte ?

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.requires_grad_

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ClassifieurTransformeur(ENCODEUR)` : l'affinage complet, avec le learning rate de 2e-5 par défaut ; `ClassifieurTransformeur(ENCODEUR, taux_apprentissage=1e-3)` pour la tête seule
- `tete_seule.modele.roberta.requires_grad_(False)` fige tous les paramètres de l'encodeur ; la tête est `tete_seule.modele.classifier`
- `sum(p.numel() for p in transformeur.parameters() if p.requires_grad)` : le nombre de paramètres entraînables, que le résumé affiché par Lightning au début de `fit` donne aussi
- la barre de progression de `fit` affiche la durée de l'epoch
- `affine` renvoie les prédictions des 1 000 premiers tweets de test ; `predictions` et `predictions_roberta` : les prédictions du GRU (partie 4) et du RoBERTa de cardiffnlp (partie 5)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
complet = ClassifieurTransformeur(ENCODEUR)
tete_seule = ClassifieurTransformeur(ENCODEUR, taux_apprentissage=...)
tete_seule.modele.roberta.requires_grad_(...)
for nom, transformeur in [("Affinage complet", complet), ("Tête seule", tete_seule)]:
  entrainables = sum(p.numel() for p in transformeur.parameters() if ...)
  print(f"{nom} : {entrainables} paramètres entraînables")

predictions_complet = affine(complet, "distilroberta_complet")
predictions_tete = affine(..., "distilroberta_tete")
for nom, predictions_modele in [
  ("GRU", predictions),
  ("DistilRoBERTa, tête seule", ...),
  ("DistilRoBERTa affiné", ...),
  ("RoBERTa de cardiffnlp", predictions_roberta),
]:
  score = recall_score(verites[:1000], predictions_modele[:1000], average="macro")
  print(f"{nom} : {score:.3f}")
```

#### Solution

In [ ]:
complet = ClassifieurTransformeur(ENCODEUR)
# Une tête qui part de zéro : le learning rate d'un modèle neuf
tete_seule = ClassifieurTransformeur(ENCODEUR, taux_apprentissage=1e-3)
# Encodeur figé : seule la tête, modele.classifier, apprend
tete_seule.modele.roberta.requires_grad_(False)
for nom, transformeur in [("Affinage complet", complet), ("Tête seule", tete_seule)]:
  entrainables = sum(p.numel() for p in transformeur.parameters() if p.requires_grad)
  print(f"{nom} : {entrainables} paramètres entraînables")

predictions_complet = affine(complet, "distilroberta_complet")
predictions_tete = affine(tete_seule, "distilroberta_tete")
affiche_evaluation(verites[:1000], predictions_complet, "DistilRoBERTa affiné")
print("Rappel moyen sur les 1 000 premiers tweets de test")
for nom, predictions_modele in [
  ("GRU", predictions),
  ("DistilRoBERTa, tête seule", predictions_tete),
  ("DistilRoBERTa affiné", predictions_complet),
  ("RoBERTa de cardiffnlp", predictions_roberta),
]:
  score = recall_score(verites[:1000], predictions_modele[:1000], average="macro")
  print(f"{nom} : {score:.3f}")

La tête seule n'a que 592 899 paramètres entraînables, contre 82 120 707 pour l'affinage complet : la rétropropagation s'arrête à la tête, et son epoch est plus courte.

Mais l'encodeur affiné l'emporte en général nettement, et s'approche du RoBERTa de cardiffnlp sans l'égaler : dans notre exécution, sur les 1 000 premiers tweets de test, un rappel moyen de 0,593 pour le GRU, 0,648 pour la tête seule, 0,710 pour l'encodeur affiné et 0,715 pour le RoBERTa de cardiffnlp. Les représentations de `distilroberta-base`, apprises pour deviner des mots masqués dans des textes généralistes, ne sont pas faites pour le sentiment des tweets : figées, la tête doit s'en contenter ; affinées, elles s'adaptent à la tâche. Le RoBERTa de cardiffnlp garde l'avantage : plus gros, il a été pré-entraîné sur des millions de tweets, dont il connaît le vocabulaire et les tournures.

Affiner tout l'encodeur est la démarche standard quand aucun modèle n'existe déjà pour la tâche et que l'on dispose de quelques milliers d'exemples annotés et d'un GPU. L'extraction de caractéristiques reste utile avec très peu de données, quand le calcul est compté, ou pour servir plusieurs tâches avec un même encodeur figé.